# 第72课 · 只动 0.5% 的参数，教会 Whisper 你的方言——LoRA 微调（fine-tuning）实战

**目标**：用 HuggingFace `Trainer` + LoRA 在 LibriSpeech-clean-100h 子集上微调
`openai/whisper-small`，在 `test-clean` 上把 WER 压到 5% 以下。

🔗 **Aurora 连接**：调用 `aurora.audio.mel.mel_spectrogram(signal, sr, n_fft=400, hop_length=160, n_mels=80)` 可与 Whisper 预处理对齐（25 ms 帧长、10 ms 步长、80 mel 通道）。注意：默认参数（`n_fft=1024, hop_length=256`）对应 64 ms / 16 ms，与 Whisper **不同**，须显式传参才能对齐；实验结论将写入 blog 第2篇 `docs/blog/02_whisper_finetune.md`。

← **上一课**　[L71 · Whisper 解码策略](L71_whisper_decoding.ipynb)

> 上节课学习了 **Whisper 解码策略**：贪婪解码与 beam search 从原理到实现。  
> 本课将探讨 **Whisper 微调**。

> ⚠️ **云环境前置检查**：Whisper-small 微调需至少 **8 GB 显存**（T4/A10 可跑通，A100/H100 更快），本地 CPU 上完整训练不现实（LibriSpeech-100h 子集约需数小时）。推荐平台：**Colab Pro+（A100）** 或 **RunPod（约 $1.99/h A100）/ Lambda Labs**，单次微调（1 h A100）成本约 $2。详细配置、断点续训与费用控制见 `docs/current/course/cloud_gpu_plan.md`。
>
> 💡 **本地也能学**：本 notebook 已对 `transformers / peft / datasets / jiwer / torch` 做**可选依赖降级**——缺失这些包时，HuggingFace 演示单元格会自动跳过并打印提示，而**纯 numpy 的练习（LoRA 参数量、`simple_wer`）与白板挑战照常运行**，无需 GPU 即可完成本课的核心学习目标。

## 学习目标

完成本节后，你应能：

1. **理解 LoRA 低秩分解原理**：推导给定 rank 下的参数量，解读 `print_trainable_parameters` 输出
2. **用 `peft` 注入 LoRA**：配置 `LoraConfig`，理解 `r`、`lora_alpha`、`target_modules` 的含义
3. **配置 `Seq2SeqTrainer`**：设置关键训练参数，理解有效 batch 计算方法
4. **计算并理解 WER**：用 `jiwer` 计算词错误率，解释 S（替换）/ D（删除）/ I（插入）含义
5. **将 Whisper 适配中文 / 方言数据**：切换语言、替换数据集、使用字符级 CER

## 本课剧情：244M 参数的模型，你只需要"改动" 0.5% 就能让它说中文

Whisper-small 有 2.44 亿个参数。全量微调（fine-tuning all parameters）：需要存储所有梯度（244M×float32 ≈ 1GB），学习率极难调，少量方言数据很容易过拟合（overfitting）。

**LoRA（Low-Rank Adaptation）**的反直觉结论：大多数深度学习任务只需要在**低秩子空间**内调整权重——你不需要改动全部 244M 参数，只需改动其中约 **0.5%（~1.2M 参数）**，效果接近全参微调。

**类比**：你要改一幅 4K 壁画的风格。全量方法：重画每个像素（244M 次操作）。LoRA 方法：只添加一层薄薄的"风格滤镜"（低秩矩阵 A、B），壁画本身冻结不动。

**LoRA 数学**：对权重矩阵 `W ∈ ℝ^{d×k}`：
```
W_new = W + ΔW = W + B·A
其中 B ∈ ℝ^{d×r}，A ∈ ℝ^{r×k}，r << min(d,k)
参数量: d×r + r×k  vs  原始 d×k（全量）
```
当 `d=k=512, r=8`：LoRA 参数 = 512×8 + 8×512 = 8192，原始 = 262144，**节省 97%**。

**本节流程**：LoRA 注入 → HuggingFace Seq2SeqTrainer → WER 评估 → 手算参数量练习。

## 🖼️ 一图看懂 rank=2：为什么"两个瘦矩阵"能改动大矩阵

冻结原权重 `W`，只训练**旁边加的一条低秩分支** `B·A`。看形状就懂省在哪：

```
        原始全量微调                     LoRA (rank r=2)
   ┌─────────────────┐          ┌─────────────────┐   ┌──┐
   │                 │          │                 │   │  │
   │        W        │          │        W        │ + │B │ ·  ┌──────────┐
   │   d × k  (冻结)  │          │   d × k  (冻结)  │   │  │    │    A     │
   │                 │          │                 │   │  │    │  r × k   │
   └─────────────────┘          └─────────────────┘   └──┘    └──────────┘
    要更新 d×k 个参数              W 不动，只训练 B(d×r) 和 A(r×k)
                                   参数量 = d·r + r·k  ≪  d·k
```

**直觉**：`B·A` 是两个瘦长矩阵相乘，结果仍是 `d×k` 形状的一块"补丁 ΔW"，但它被约束在 **rank ≤ r** 的低维子空间里。就像给一幅大画贴一层薄滤镜——用极少的自由度，去"微调"而非"重画"整个 `W`。

以 `d=k=768, r=8` 为例：全量 `768² ≈ 590K`，LoRA `2×768×8 ≈ 12K`——**同一处权重，参数少了 ~98%**。（本课下面的练习会让你亲手算这笔账。）


In [ ]:
# 需要 cloud GPU；依赖: transformers, peft, datasets, jiwer
# pip install transformers peft datasets jiwer
# 这些是可选依赖：缺失时本课的纯 numpy 练习与白板仍可运行（见开头「云环境前置检查」）。
try:
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
    from peft import LoraConfig, get_peft_model
    HAS_HF = True
except ImportError:
    HAS_HF = False
    print("⏭️ 未安装 transformers/peft（本机 CPU 环境）。")
    print("   下方 HuggingFace 演示单元格将自动跳过；纯 numpy 练习与白板照常运行。")
    print("   云端配置见 docs/current/course/cloud_gpu_plan.md")

## 1. LoRA（Low-Rank Adaptation）

对于权重矩阵 `W ∈ R^{d × k}`，LoRA 冻结 `W`，额外引入：

```
ΔW = A @ B       A ∈ R^{d × r},  B ∈ R^{r × k},  r << min(d, k)
```

前向传播变为 `y = (W + ΔW) x = W x + A @ B @ x`。初始化时 `A ~ N(0, σ²)`、`B = 0`，保证训练起点与原始模型等价。

对 Whisper-small（`d_model = 768`），自注意力中 `q_proj` 和 `v_proj` 各为 `768 × 768`。取 `r = 8`：原始可训练参数约 `2 × 768² ≈ 1.18 M`；LoRA 参数 `2 × 2 × 768 × 8 = 24 576`，**降幅 98%**。


In [ ]:
# 演示：加载模型并注入 LoRA，打印可训练参数比例
if HAS_HF:
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,          # 缩放因子：ΔW 乘以 lora_alpha / r = 4
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # print_trainable_parameters() 会在全部匹配到的 q_proj/v_proj 模块上
    # （编码器自注意力 + 解码器自注意力 + 解码器交叉注意力，共几十个矩阵）注入
    # A/B 低秩分支，可训练参数占比通常远低于 1%（trainable% < 1）。
else:
    print("⏭️ 跳过 LoRA 注入演示（需 transformers/peft，且会下载 ~1 GB 模型权重）。")
    print("   概念要点：get_peft_model 冻结原权重，仅在 q_proj/v_proj 上加 B·A 低秩分支。")

## 2. HuggingFace `Seq2SeqTrainer`

`Seq2SeqTrainer` 在 `Trainer` 基础上增加 **生成式评估**（beam search），适配 encoder-decoder 模型。关键 `TrainingArguments` 参数：

| 参数 | 含义 |
|---|---|
| `per_device_train_batch_size` | 每卡 batch，影响显存；常取 16 |
| `gradient_accumulation_steps` | 累积 N 步再更新，等效扩大 batch |
| `fp16` | 混合精度；A100/V100 可开 |
| `predict_with_generate` | eval 时调用 `model.generate()` 算真实 WER |
| `generation_max_length` | 解码最长 token 数（LibriSpeech 句子取 225）|
| `save_steps` / `eval_steps` | 每 N 步存 checkpoint / 跑 eval |

实际有效 batch = `per_device_train_batch_size × gradient_accumulation_steps × num_gpus`。单卡 A100 40 GB 时，`batch=16, accumulation=2` 等效 batch=32。


In [ ]:
# 演示：加载数据集 + Processor，构造 DataCollator，配置 TrainingArguments
if HAS_HF:
    from datasets import load_dataset
    from transformers import DataCollatorForSeq2Seq
    import torch

    processor = WhisperProcessor.from_pretrained(
        "openai/whisper-small", language="English", task="transcribe"
    )

    # 只下载 train-clean-100（约 6 GB）和 test-clean 做快速演示
    # 完整训练时去掉 split 限制
    dataset = load_dataset(
        "librispeech_asr", "clean",
        split={"train": "train.clean.100", "test": "test.clean"},
        trust_remote_code=True,
    )

    def preprocess(batch):
        audio = batch["audio"]
        batch["input_features"] = processor(
            audio["array"], sampling_rate=audio["sampling_rate"],
            return_tensors="pt"
        ).input_features[0]
        batch["labels"] = processor.tokenizer(batch["text"]).input_ids
        return batch

    dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names, num_proc=4)

    data_collator = DataCollatorForSeq2Seq(processor.tokenizer, model=model, padding=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir="./whisper-small-lora-librispeech",
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,    # 有效 batch = 32
        learning_rate=1e-4,
        num_train_epochs=3,
        fp16=torch.cuda.is_available(),
        predict_with_generate=True,
        generation_max_length=225,
        save_steps=500,
        eval_steps=500,
        eval_strategy="steps",      # transformers>=4.38 新名称（旧名 evaluation_strategy 已弃用）
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,          # WER 越低越好
        report_to="tensorboard",
    )
    print("TrainingArguments 配置完成")
else:
    print("⏭️ 跳过数据集加载与 TrainingArguments 配置（需 datasets/transformers，且会下载 ~6 GB LibriSpeech）。")
    print("   有效 batch = per_device_train_batch_size × gradient_accumulation_steps × num_gpus。")

## 3. 评估指标：词错误率（WER）

> **L72/L73 衔接**：本节介绍 WER 背景与计算方法；L73 将在此基础上深入实现编辑距离回溯，逐句统计 S / D / I 比率并定位高 WER 句子，不再重复基础定义。

```
WER = (S + D + I) / N
```

- `S`：替换（substitution）；`D`：删除（deletion）；`I`：插入（insertion）
- `N`：参考文本总词数

`jiwer` 库用动态规划在毫秒内算出编辑距离。LibriSpeech-test-clean 是朗读体英语，基线（zero-shot Whisper-small）约 **4–5% WER**；在 100h 子集上微调后目标 **< 5%**，甚至可逼近 2–3%。

注意：WER 对大小写和标点敏感，所以算之前要先做 **normalization**（`WhisperProcessor` 内置 `normalizer`）。


In [ ]:
# 演示：用 jiwer 计算 WER + 配置 compute_metrics 回调
try:
    import jiwer
    HAS_JIWER = True
except ImportError:
    HAS_JIWER = False

# compute_metrics 需要 jiwer + HuggingFace processor/normalizer
if HAS_JIWER and HAS_HF:
    from transformers.models.whisper.english_normalizer import BasicTextNormalizer

    normalizer = BasicTextNormalizer()

    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        # 把 -100（padding）替换为 pad_token_id
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
        # 归一化
        pred_str  = [normalizer(s) for s in pred_str]
        label_str = [normalizer(s) for s in label_str]
        wer = jiwer.wer(label_str, pred_str)
        return {"wer": wer}

# 快速单元测试：WER 计算是否正确（只需 jiwer）
if HAS_JIWER:
    ref  = ["the cat sat on the mat"]
    hyp  = ["the cat sat on a mat"]          # 1 次替换，共 6 词
    wer_val = jiwer.wer(ref, hyp)
    assert abs(wer_val - 1/6) < 1e-6, f"WER 计算错误: {wer_val}"
    print(f"✅ WER 单元测试通过：WER = {wer_val:.4f}（期望 {1/6:.4f}）")
else:
    print("⏭️ 未安装 jiwer，跳过 WER 演示；下方纯 numpy 的 simple_wer 练习不受影响。")

## ✏️ 练习：手算 LoRA 参数量 + 实现简化 WER

### 练习 1：参数量计算

给定以下配置（对应下方代码单元格的变量）：
- `d_model = 512`，`r = 16`（LoRA rank）
- 一层注意力有 `q_proj`、`k_proj`、`v_proj` 三个 `512×512` 矩阵（`num_matrices = 3`）

**手算路线**：
| 步骤 | 公式 | 结果 |
|---|---|---|
| 单矩阵原始参数 | `d×d` = `512×512` | 262144 |
| 单矩阵 LoRA 参数 | `d×r + r×d` = `2×512×16` | 16384 |
| 3 个矩阵 LoRA 合计 | `3 × 16384` | 49152 |
| 3 个矩阵全参 baseline | `3 × 512²` | 786432 |
| 降幅 | `(1 − 49152/786432) × 100%` | **≈ 93.75%** |

> 换 `r = 8` 或只注入 q/v 两个矩阵，降幅会更大——这正是本课演示配置只挑 `q_proj`/`v_proj` 的原因。

### 练习 2：实现简化 WER

**签名**：
```python
def simple_wer(ref: str, hyp: str) -> float:
    # ref: 参考文本（ground truth）
    # hyp: 模型输出文本（hypothesis）
    # 返回: WER ∈ [0.0, ∞)，WER=(S+D+I)/N
```

**实现路线**：用编辑距离。将 `ref` 和 `hyp` 按空格分词，计算词级编辑距离（与 L67/L68 的 `edit_distance()` 相同），除以 ref 单词数。

**验收标准**：
- `simple_wer("hello world", "hello world")` == 0.0
- `simple_wer("hello world", "hello")` == 0.5（删除1个词，共2个词）
- `simple_wer("a b c", "x b c")` ≈ 0.333（替换1个词）

In [ ]:
# ─── 练习 1：LoRA 参数量计算 ───────────────────────────────────────────
d_model = 512
r = 16
num_matrices = 3  # q_proj, k_proj, v_proj

def lora_param_math(d_model: int, r: int, num_matrices: int):
    """手算 LoRA 参数量，返回 (total_lora_params, total_original_params, reduction_pct)。

    把下面每个 raise 逐行替换成正确的赋值表达式（公式见上方"手算路线"表），
    最后取消注释 return 行。
    """
    raise NotImplementedError("TODO: 计算 original_params_per_matrix = d_model * d_model")
    raise NotImplementedError("TODO: 计算 lora_params_per_matrix = 2 * d_model * r（A + B 矩阵）")
    raise NotImplementedError("TODO: 计算 total_lora_params = num_matrices * lora_params_per_matrix")
    raise NotImplementedError("TODO: 计算 total_original_params = num_matrices * original_params_per_matrix")
    raise NotImplementedError("TODO: 计算 reduction_pct = (1 - total_lora_params / total_original_params) * 100")
    # return total_lora_params, total_original_params, reduction_pct

# 验证（实现后自动运行；未实现时优雅跳过，不中断 Run All）
try:
    total_lora_params, total_original_params, reduction_pct = lora_param_math(
        d_model, r, num_matrices
    )
    expected_lora = 3 * 2 * 512 * 16  # = 49152
    expected_reduction = (1 - expected_lora / (3 * 512 * 512)) * 100  # ≈ 93.75%
    assert total_lora_params == expected_lora, f'参数量错误: {total_lora_params}'
    assert abs(reduction_pct - expected_reduction) < 0.01, f'降幅错误: {reduction_pct:.2f}%'
    print(f'✅ LoRA 参数量: {total_lora_params:,}，降幅 {reduction_pct:.1f}%')
except (NotImplementedError, TypeError):
    print('⏳ 尚未完成练习 1（lora_param_math），完成 TODO 后重新运行')

# ─── 练习 2：不依赖 jiwer 的简化 WER ────────────────────────────────────
def simple_wer(ref: str, hyp: str) -> float:
    """单句词级 WER（编辑距离 / 参考词数），不使用 jiwer。
    ref / hyp 为普通字符串，内部按空格分词。
    hint: 用动态规划计算 ref 与 hyp 分词后的编辑距离。
    """
    # TODO: 实现
    raise NotImplementedError('请实现 simple_wer')

# 验证（实现后运行）
try:
    result = simple_wer("the cat sat on the mat", "the cat sat on a mat")
    assert abs(result - 1/6) < 1e-6, f'WER 错误: {result}'
    print(f'✅ simple_wer 通过: WER = {result:.4f}')
except (NotImplementedError, TypeError):
    print('⏳ 尚未实现 simple_wer，完成 TODO 后重新运行')

## 参数实验：监控训练曲线，定位 overfitting

**可调参数与预期现象**：

| 参数 | 建议值 | 预期现象 |
|---|---|---|
| `r`（LoRA rank）| 4 / 8 / 16 | r 越大表达力越强，但过拟合风险更高；r=8 通常最优 |
| `lora_alpha` | `2r` 或 `4r` | alpha/r 是实际缩放倍数，增大等效提高学习率 |
| `num_train_epochs` | 3–5 | epoch 3 后 eval WER 通常不再下降，train loss 继续降→过拟合 |
| `learning_rate` | 1e-4 / 5e-5 | 1e-4 收敛快；5e-5 更稳但需更多 epoch |
| `lora_dropout` | 0.0 / 0.05 | dropout=0.05 对小数据集有正则效果 |

**监控方式**：TensorBoard 或 `trainer.state.log_history` 中的 `loss` 和 `eval_wer`。当 `eval_wer` 连续 2 个 eval_step 不下降而 `loss` 仍在降，就是 overfitting 的开始。


## 4. 中文 / 方言数据适配实战

Whisper 通过 `WhisperProcessor` 的 `language` 参数切换目标语言。中文识别与英文的主要差异：

| 差异点 | 英文（LibriSpeech） | 中文（AISHELL-1 / Common Voice zh）|
|---|---|---|
| 分词粒度 | 空格分词 | 字符级（无空格）|
| 评估指标 | 词错误率 WER | 字符错误率 CER（Character Error Rate）|
| 文本归一化 | `BasicTextNormalizer` | 去标点、繁简转换 |
| 数据集 | `librispeech_asr` | `mozilla-foundation/common_voice_13_0` `zh-CN` 或 `aishell` |
| Processor 语言 | `language="English"` | `language="Chinese"` |

> **字符错误率 CER**：计算前将所有空格去掉，再做字符级编辑距离，公式与 WER 相同但分词粒度为单个汉字。

In [ ]:
# 演示：中文 / 方言 Whisper 微调适配
# 实际训练需有 Common Voice zh-CN 或 AISHELL-1 数据集访问权限

# 步骤 1：切换 Processor 语言至中文（需 transformers）
if HAS_HF:
    processor_zh = WhisperProcessor.from_pretrained(
        "openai/whisper-small",
        language="Chinese",   # 切换为中文
        task="transcribe"
    )
    print("✅ processor_zh 已切换为中文")
else:
    print("⏭️ 跳过中文 Processor 加载（需 transformers）")

# 步骤 2：字符级 CER 计算（去空格，按字符计算编辑距离）
if HAS_JIWER:
    def compute_cer(reference: str, hypothesis: str) -> float:
        """字符级错误率：拆为字符列表后利用 jiwer 计算 WER。"""
        ref_chars = list(reference.replace(' ', ''))
        hyp_chars = list(hypothesis.replace(' ', ''))
        return jiwer.wer(' '.join(ref_chars), ' '.join(hyp_chars))

    # 单元测试
    ref_zh = "我在北京学习机器学习"   # 10 个汉字
    hyp_zh = "我在北京学习机器语言"  # 末尾两字错误（2/10 字符）
    cer_val = compute_cer(ref_zh, hyp_zh)
    expected_cer = 2 / 10
    assert abs(cer_val - expected_cer) < 1e-6, f'CER 计算错误: {cer_val:.4f}，期望 {expected_cer:.4f}'
    print(f'✅ CER 单元测试通过: CER = {cer_val:.4f}（2/10 字符错误）')
else:
    print("⏭️ 未安装 jiwer，跳过 CER 演示（去空格后按字符做词级编辑距离即得 CER）")

# 步骤 3：数据集替换（需实际访问权限）
# dataset_zh = load_dataset(
#     "mozilla-foundation/common_voice_13_0", "zh-CN",
#     split={"train": "train", "test": "test"},
#     trust_remote_code=True,
# )
# 或：dataset_zh = load_dataset('aishell', split='train')

# 步骤 4：中文 compute_metrics 回调（字符级 CER，需 transformers + jiwer）
if HAS_HF and HAS_JIWER:
    def compute_metrics_zh(pred):
        pred_ids  = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = processor_zh.tokenizer.pad_token_id
        pred_str  = processor_zh.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor_zh.batch_decode(label_ids, skip_special_tokens=True)
        cer = sum(compute_cer(r, h) for r, h in zip(label_str, pred_str)) / len(label_str)
        return {'cer': cer}

    print('✅ 中文适配组件准备完毕：processor_zh / compute_cer / compute_metrics_zh')
    print('将 compute_metrics 替换为 compute_metrics_zh，并在 TrainingArguments 中改 metric_for_best_model="cer" 即可用于中文微调')
else:
    print('（云环境安装 transformers + jiwer 后，可组装 compute_metrics_zh 用于中文微调）')

In [ ]:
# 启动训练（需 GPU；在 CPU 上会极慢，仅作流程验证）
if HAS_HF:
    import torch
    if torch.cuda.is_available():
        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["test"],
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            tokenizer=processor.tokenizer,    # transformers<4.40: tokenizer=; newer: processing_class=processor
        )
        trainer.train()
        # 训练结束后查看最佳 checkpoint
        print("最佳 checkpoint：", trainer.state.best_model_checkpoint)
        print("最佳 WER：", trainer.state.best_metric)
    else:
        print("⚠️  未检测到 GPU，跳过训练。请在 cloud GPU 环境（如 Colab A100 / RunPod A100）运行。")
else:
    print("⏭️ 未安装 transformers/torch，跳过训练演示。")
    print("   云 GPU 配置、断点续训与费用控制见 docs/current/course/cloud_gpu_plan.md")

# 可选：绘制 loss / WER 曲线
# import pandas as pd, matplotlib.pyplot as plt
# log_df = pd.DataFrame(trainer.state.log_history)
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# log_df.dropna(subset=["loss"]).plot("step", "loss", ax=axes[0], title="Train Loss")
# log_df.dropna(subset=["eval_wer"]).plot("step", "eval_wer", ax=axes[1], title="Eval WER")
# plt.tight_layout(); plt.show()

## 🎯 未来的回报 (Future Payoff)

今天你亲手搞懂的 **LoRA = 冻结 W + 只训练低秩 B·A**，会在 **L84（LoRA 从零实现）** 再次出现——那节课你不再调 `peft`，而是**亲手写出** `LoRALinear` 的前向与反向，把今天在 Whisper 上"用"的东西，变成"造"出来的东西。

同一个低秩思想还会照进：
- **L73 / L74**：微调后的模型好不好，靠 WER 的 S/D/I 逐句诊断来量化——LoRA 只是"改"，评估才知道"改对没有"。
- **参数高效微调（PEFT）大家族**：QLoRA、Adapter、Prefix-tuning 都是"冻结主干、只训练一小撮参数"的变体，今天这张 `d·r + r·k ≪ d·k` 的账，是它们共同的地基。

值得记牢的一句：**大模型时代，"改得动"往往不靠参数多，而靠改在对的低维子空间里。**


## 本课收束

`get_peft_model` 注入 LoRA 后，`Seq2SeqTrainer` 只更新 `q_proj`/`v_proj` 的低秩矩阵，在 3 个 epoch 内即可在 LibriSpeech-test-clean 上输出 WER < 5% 的转录结果。Aurora 模块 `aurora.audio.mel.mel_spectrogram` 的 mel 预处理与`WhisperProcessor` 完全对齐，可复用于离线特征缓存以加速数据管道。下一节（L73）将计算真实转录的词错率（WER）：实现编辑距离、统计 S/D/I 各类错误比率，并定位 WER 最高的句子。


## ✏️ 白板挑战：LoRA 原理手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：LoRA 为什么用两个矩阵 B·A 而不是一个低秩矩阵 C？  
（初始化：B=0，A从N(0,σ²)随机初始化→初始ΔW=B·A=0，不影响预训练权重；直接用C无法保证零初始化）

**问 2**：若 d=1024，k=1024，r=4，LoRA 参数量是多少？全参数量是多少？节省比例？  
（LoRA: 1024×4+4×1024=8192；全参: 1024²=1,048,576；节省=1-8192/1048576≈99.2%）

**问 3**：为什么 Whisper 微调通常只在 q_proj 和 v_proj 注入 LoRA，而不是所有权重？  
（注意力的query/value决定"关注什么"，是任务适配的关键；其余层（k_proj,out,FFN）改动收益递减且参数更少，实践发现q+v足够）

**问 4**：WER > 1.0 是否可能？什么情况下发生？  
（可能：插入词极多时，如ref="good"(1词), hyp="this is very very good"(5词)→WER=(0+0+4)/1=4.0）

**问 5**：LoRA 推理时是否会增加延迟？为什么？  
（理论上不增加：推理前将B·A合并回W（W_new=W+B·A），参数数量不变，延迟为零；训练时用分离形式是为了只更新A/B梯度）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格

# 问1：LoRA零初始化（概念验证）
import numpy as np
d, k, r = 512, 512, 8
B = np.zeros((d, r))
A = np.random.randn(r, k) * 0.01
delta_W = B @ A
assert np.allclose(delta_W, 0), "初始ΔW=B·A应全为0（B零初始化）"
print(f"Q1 ✅  初始ΔW=B·A: max|ΔW|={np.abs(delta_W).max():.2e}（全零，不影响预训练权重）")

# 问2：参数量计算
d2, k2, r2 = 1024, 1024, 4
lora_params = d2 * r2 + r2 * k2
full_params = d2 * k2
savings_pct = (1 - lora_params / full_params) * 100
assert lora_params == 8192, f"LoRA参数={lora_params}，期望8192"
assert full_params == 1048576, f"全参={full_params}，期望1048576"
print(f"Q2 ✅  d={d2},k={k2},r={r2}: LoRA={lora_params:,} vs 全参={full_params:,}，节省{savings_pct:.1f}%")

# 问3：Whisper LoRA范围（概念验证 — 通过实际参数比计算）
d_ws, r_ws, n_layers = 512, 8, 12
lora_qv = n_layers * 2 * (d_ws * r_ws + r_ws * d_ws)
full_attn_qkvo = n_layers * 4 * d_ws * d_ws
print(f"Q3 ✅  q+v LoRA参数={lora_qv:,}（共{n_layers}层），占全量注意力参数{full_attn_qkvo:,}的{lora_qv/full_attn_qkvo*100:.1f}%")

# 问4：WER > 1.0 验证
try:
    wer_val = simple_wer("good", "this is very very good")
    assert wer_val > 1.0, f"WER={wer_val:.2f}，插入4词应>1.0"
    print(f"Q4 ✅  simple_wer('good', 'this is very very good')={wer_val:.2f}>1.0（插入词多时WER>100%）")
except (NotImplementedError, NameError):
    # 手动验证
    ref_words = ["good"]
    hyp_words = ["this", "is", "very", "very", "good"]
    # edit_distance = 4 insertions, N = 1
    manual_wer = 4.0 / 1
    print(f"Q4 ✅  WER='good'vs'this is very very good'≈{manual_wer:.1f}>1（待实现simple_wer后验证）")

# 问5：LoRA推理延迟（数值验证合并操作）
W = np.random.randn(d, k) * 0.1
B_trained = np.random.randn(d, r) * 0.01
A_trained = np.random.randn(r, k) * 0.01
W_merged = W + B_trained @ A_trained  # 推理时合并
x = np.random.randn(k)
out_lora = W @ x + (B_trained @ (A_trained @ x))  # LoRA分离（训练时）
out_merged = W_merged @ x                          # 合并后（推理时）
assert np.allclose(out_lora, out_merged, atol=1e-10), "合并前后输出应完全相同"
print(f"Q5 ✅  合并验证：max|out_lora - out_merged|={np.abs(out_lora-out_merged).max():.2e}（推理零延迟增加）")
print("\n🎉 LoRA原理白板挑战通过！")

In [ ]:
try:
    # ✏️ 本课自评
    l72_review = {
        "lora_math_understood":     None,  # 理解LoRA: W_new=W+B·A，B零初始化，只训练A/B？True/False
        "param_count_derivation":   None,  # 能手算给定d/k/r时的LoRA参数量和节省比例？True/False
        "simple_wer_impl":          None,  # simple_wer()实现正确，3个验收标准全通过？True/False
        "trainer_pipeline":         None,  # 理解Seq2SeqTrainer流程：DataCollator→TrainingArgs→train()→evaluate()？True/False
        "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
    }

    unfilled = [k for k, v in l72_review.items() if v is None]
    assert not unfilled, f'还未填写：{unfilled}'
    weak = [k for k, v in l72_review.items() if v is False]
    if weak:
        print(f'⚠️  需要加强：{weak}')
    else:
        print('✅ L72 全部通关！进入 L73：WER 评估')
except (NotImplementedError, TypeError):
    print('⬜ 请先完成上面的 TODO 练习，再运行本格')

---

→ **下一课**　[L73 · WER 评估](L73_wer_eval.ipynb)

> 下节课将学习 **WER 评估**：词错误率（插入 / 删除 / 替换），jiwer 对比逐句分析。